[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/somanshusingla/llm-from-scratch/blob/main/notebooks/01_ch2_working_with_text_data.ipynb)

# Chapter 2 — Working with Text Data

**Build a Large Language Model (From Scratch)** · notebook 1 of 7

> **New to all this?** Great — this notebook assumes *zero* prior deep-learning
> experience. Every code cell is short and has an explanation above it. Run the
> cells one by one (Shift+Enter) and read as you go.

### What an LLM actually eats
A language model cannot read raw text. It only understands **numbers**. So the
very first job — before any "AI" happens — is turning a string like
`"The verdict was clear"` into a list of numbers the model can process.

This chapter builds that whole pipeline, step by step:

1. **Tokenize** — split text into small pieces ("tokens").
2. **Token IDs** — map every unique token to an integer.
3. **Special tokens** — handle unknown words and document boundaries.
4. **Byte-Pair Encoding (BPE)** — the real tokenizer GPT-2 uses.
5. **Sliding-window sampling** — turn a long text into (input, target) training pairs.
6. **Embeddings** — turn token IDs into learnable vectors.
7. **Positional embeddings** — tell the model *where* each token sits.

By the end you'll have the exact `input embeddings` tensor that Chapter 3's
attention mechanism consumes.

---
*Runs on a laptop CPU in seconds. Also runs as-is on Google Colab.*

## 0. Setup

If you're on **Google Colab** or a fresh machine, the next cell installs the one
library that isn't built in: `tiktoken` (OpenAI's fast BPE tokenizer).
On the local `venv` used to test these notebooks it's already installed, so this
is a no-op there.

In [ ]:
# Install only what's missing (safe to re-run).
import importlib, subprocess, sys

def ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f"Installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

ensure("tiktoken")
print("Setup OK")

Setup OK


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import tiktoken

print("torch:", torch.__version__)
print("tiktoken:", tiktoken.__version__)

torch: 2.11.0+cu128
tiktoken: 0.13.0


## 2.1 Understanding word embeddings

Deep-learning models work with vectors of real numbers, not text. An
**embedding** maps a discrete thing (a word/token) to a continuous vector, e.g.
the token `"cat"` → `[0.12, -0.98, 0.33, ...]`.

We can't feed words directly, so we first convert text → token IDs (integers),
then look up a vector for each ID in an *embedding table*. The embedding table is
just a big matrix of numbers that the model **learns** during training.

We'll get to the embedding lookup in section 2.7. First we need token IDs.

## 2.2 Tokenizing text

Let's get some real text. We use *"The Verdict"*, a short public-domain story by
Edith Wharton — the same text the book uses. The cell downloads it, and if you're
offline it falls back to a small built-in sample so the notebook still runs.

In [ ]:
import os, urllib.request

file_path = "the-verdict.txt"
url = ("https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/"
       "ch02/01_main-chapter-code/the-verdict.txt")

if not os.path.exists(file_path):
    try:
        urllib.request.urlretrieve(url, file_path)
        print("Downloaded", file_path)
    except Exception as e:
        print("Download failed (", e, ") -- writing a small fallback sample.")
        fallback = (
            "I HAD always thought Jack Gisburn rather a cheap genius--though a good "
            "fellow enough--so it was no great surprise to me to hear that, in the "
            "height of his glory, he had dropped his painting, married a rich widow, "
            "and established himself in a villa on the Riviera. " * 40
        )
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(fallback)

with open(file_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total number of characters:", len(raw_text))
print(raw_text[:120])

Downloaded the-verdict.txt
Total number of characters: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me 


Now split the text into tokens. A first idea: split on whitespace and
punctuation using a regular expression. This keeps punctuation as its own tokens
(commas, periods, etc.), which matters for language.

In [ ]:
import re

# Split on whitespace AND punctuation, keeping the punctuation as separate tokens.
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]

print("Number of tokens:", len(preprocessed))
print(preprocessed[:30])

Number of tokens: 4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


## 2.3 Converting tokens into token IDs

To give each token an integer ID, we build a **vocabulary**: the sorted set of
all unique tokens, each assigned an index.

In [ ]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print("Vocabulary size:", vocab_size)

vocab = {token: integer for integer, token in enumerate(all_words)}
# Peek at the first few (token -> id) mappings
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 9:
        break

Vocabulary size: 1130
('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)


Let's wrap encode/decode in a small tokenizer class. `encode` turns text into IDs;
`decode` turns IDs back into text.

In [ ]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)  # fix spaces before punctuation
        return text


tokenizer = SimpleTokenizerV1(vocab)
sample = "It was the last he painted, you know."
ids = tokenizer.encode(sample)
print("IDs:   ", ids)
print("Back:  ", tokenizer.decode(ids))

IDs:    [56, 1077, 988, 602, 533, 746, 5, 1126, 596, 7]
Back:   It was the last he painted, you know.


## 2.4 Adding special context tokens

The tokenizer above **crashes** on any word it hasn't seen. Real models handle
this with special tokens. We add two:

- `<|unk|>` — stands in for unknown tokens.
- `<|endoftext|>` — marks the boundary between independent documents.

In [ ]:
all_tokens = sorted(set(preprocessed))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token: integer for integer, token in enumerate(all_tokens)}
print("New vocab size:", len(vocab))


class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        # replace unknown words with <|unk|>
        preprocessed = [item if item in self.str_to_int else "<|unk|>"
                        for item in preprocessed]
        return [self.str_to_int[s] for s in preprocessed]

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text


tokenizer = SimpleTokenizerV2(vocab)
text = "<|endoftext|> ".join(("Hello, do you like tea?", "In the sunlit palaces."))
print(text)
print(tokenizer.encode(text))
print(tokenizer.decode(tokenizer.encode(text)))

New vocab size: 1132
Hello, do you like tea?<|endoftext|> In the sunlit palaces.
[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 1131, 7]
<|unk|>, do you like tea? <|endoftext|> In the sunlit <|unk|>.


Notice `"Hello"` and `"palaces"` (not in our tiny vocab) became `<|unk|>`. A
whole-word vocabulary is brittle. The fix used by real LLMs is **subword**
tokenization — up next.

## 2.5 Byte-Pair Encoding (BPE) — the real GPT-2 tokenizer

BPE never says "unknown". It breaks rare words into smaller known sub-pieces
(down to single bytes if needed), so *any* string can be encoded. GPT-2 uses a
BPE vocabulary of **50,257** tokens. We use OpenAI's `tiktoken` implementation.

In [ ]:
bpe = tiktoken.get_encoding("gpt2")

text = ("Hello, do you like tea? <|endoftext|> In the sunlit terraces "
        "of someunknownPlace.")
integers = bpe.encode(text, allowed_special={"<|endoftext|>"})
print("Token IDs:", integers)
print("Decoded:  ", bpe.decode(integers))
print("Vocab size:", bpe.n_vocab)

Token IDs: [15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]
Decoded:   Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.
Vocab size: 50257


In [ ]:
# BPE splits an unknown word into subword pieces -- no <|unk|> needed.
word = "Akwirw ier"
ids = bpe.encode(word)
print(ids)
for i in ids:
    print(i, "->", repr(bpe.decode([i])))

[33901, 86, 343, 86, 220, 959]
33901 -> 'Ak'
86 -> 'w'
343 -> 'ir'
86 -> 'w'
220 -> ' '
959 -> 'ier'


## 2.6 Data sampling with a sliding window

To train an LLM we need **(input, target)** pairs. The task is simply:
*given the tokens so far, predict the next token.* So the target is just the
input shifted right by one position.

We slide a window of length `max_length` across the token stream with a given
`stride` to create many training examples.

In [ ]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        # sliding window
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128,
                         shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle,
        drop_last=drop_last, num_workers=num_workers)  # num_workers=0 is safest on Colab/Windows
    return dataloader

In [ ]:
# One example at a time to see the input -> target shift clearly.
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print("Inputs: ", first_batch[0])
print("Targets:", first_batch[1])
print("(target is the input shifted one step to the right)")

Inputs:  tensor([[  40,  367, 2885, 1464]])
Targets: tensor([[ 367, 2885, 1464, 1807]])
(target is the input shifted one step to the right)


In [ ]:
# A realistic batch: 8 sequences of length 4.
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)
inputs, targets = next(iter(dataloader))
print("Inputs shape:", inputs.shape)   # (batch_size, max_length)
print(inputs)

Inputs shape: torch.Size([8, 4])
tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])


## 2.7 Creating token embeddings

Now we turn token IDs into vectors using `torch.nn.Embedding` — a lookup table
with one learnable row per vocabulary entry. Initially the numbers are random;
training will shape them into meaningful representations.

In [ ]:
torch.manual_seed(123)

vocab_size = 50257     # GPT-2 BPE vocab
output_dim = 256       # embedding size we'll use for the demo

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print("Embedding table shape:", token_embedding_layer.weight.shape)

# Embed a real batch (8 x 4 token IDs -> 8 x 4 x 256 vectors)
token_embeddings = token_embedding_layer(inputs)
print("Token embeddings shape:", token_embeddings.shape)

Embedding table shape: torch.Size([50257, 256])
Token embeddings shape: torch.Size([8, 4, 256])


## 2.8 Encoding word positions

A plain embedding table gives the *same* vector for a token no matter where it
appears. But order matters ("dog bites man" != "man bites dog"). GPT-2 adds a
second, learnable **positional embedding** — one vector per position — to each
token embedding.

`input embeddings = token embeddings + positional embeddings`

In [ ]:
max_length = 4
context_length = max_length

pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print("Positional embeddings shape:", pos_embeddings.shape)  # (4, 256)

# Broadcasting adds the (4,256) position vectors to every sequence in the batch.
input_embeddings = token_embeddings + pos_embeddings
print("Input embeddings shape:", input_embeddings.shape)     # (8, 4, 256)

Positional embeddings shape: torch.Size([4, 256])
Input embeddings shape: torch.Size([8, 4, 256])


## Summary

You just built the full input pipeline of a GPT:

`raw text` → **tokenize** → **token IDs** (BPE) → **sliding-window (input, target)
pairs** → **token embeddings** + **positional embeddings** → `input embeddings`.

That `input embeddings` tensor of shape `(batch, tokens, emb_dim)` is exactly
what the attention mechanism consumes next.

### Try it yourself
- Change `max_length` and `stride` in `create_dataloader_v1` and watch the number
  of training pairs change.
- Encode your own sentence with the BPE tokenizer and inspect the sub-pieces.

**Next:** `02_ch3_attention_mechanisms.ipynb` — the heart of the transformer.